In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression


In [ ]:
# Run from the repository root.
df = pd.read_csv("data/churn-analysis.csv", sep=";")


In [ ]:
df

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
df_number_corr = df.corr(numeric_only=True, method='pearson')

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_number_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)

In [ ]:
sns.set_style('whitegrid')
xs=sns.countplot(data=df, x='churn')
#xs.set_xlabel("")
xs.set_ylabel("")
for container in xs.containers:
    xs.bar_label(container, padding=3)

In [ ]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()

In [ ]:
for var in numeric_columns:
    sns.displot(
        data=df,
        x=var,
        hue='churn',
           
    )
    plt.title(f"{var}")

    plt.show()

In [ ]:
df.drop(columns=['area code', 'phone number'], inplace=True)

In [ ]:
X=df.drop(['churn'], axis=1)
y = df['churn'].map({True:1, False:0})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)


In [ ]:
numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(include="object").columns

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='mean')),
    ("scaler", StandardScaler())])


categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("encoder", OneHotEncoder(handle_unknown='ignore'))])



preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## LOGISTIC REGRESSION


In [ ]:
logistic_model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

lm_scores = cross_val_score(
    logistic_model_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

In [ ]:
print("Cross-validation scores:", lm_scores)
print(f"Mean AUC: {lm_scores.mean():.4f} +/-{lm_scores.std():.4f}")

## RANDOM FOREST

In [ ]:
random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
        n_estimators=300,
        max_depth=5,

         ))
])

In [ ]:
rf_scores = cross_val_score(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

In [ ]:
print("Cross-validation scores:", rf_scores)
print(f"Mean AUC: {rf_scores.mean():.4f} +/-{rf_scores.std():.4f}")

## XGBOOST

In [ ]:
from xgboost import XGBClassifier

xgb_cv_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
         n_jobs=-1, scale_pos_weight=5.9, random_state=42
    ))
])

In [ ]:
xgb_scores = cross_val_score(
    xgb_cv_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc")

In [ ]:
print("Cross-validation scores:", xgb_scores)
print(f"Mean AUC: {xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})")

## Model Comparison and Selection (Training Data Only)

Compare models using **mean stratified cross-validation ROC-AUC** on `X_train`. This metric measures discrimination without requiring a fixed classification threshold. Model selection precedes the final holdout evaluation.

In [ ]:
model_comparison_cv = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "AUC CV Mean": [
        lm_scores.mean(),
        rf_scores.mean(),
        xgb_scores.mean()
    ],
    "AUC CV Std": [
        lm_scores.std(),
        rf_scores.std(),
        xgb_scores.std()
    ]
})

model_comparison_cv = model_comparison_cv.sort_values(
    "AUC CV Mean",
    ascending=False
)

display(model_comparison_cv.round(4))


## Decision Threshold Optimization (Training Data Only)

Generate **out-of-fold** predicted probabilities using `cross_val_predict` on `X_train`, then select the threshold that maximizes F1 for the positive class. The test set is **not used** for threshold tuning.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

# Generate out-of-fold probabilities
y_proba_cv = cross_val_predict(
    xgb_cv_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)[:, 1]

# Evaluate candidate thresholds
thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:

    y_pred = (y_proba_cv >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, y_pred, zero_division=0),
        "Recall": recall_score(y_train, y_pred, zero_division=0),
        "F1": f1_score(y_train, y_pred, zero_division=0)
    })

# Crear DataFrame
df_thresholds = pd.DataFrame(threshold_results)

# Encontrar threshold que maximiza F1
best_idx = df_thresholds["F1"].idxmax()

best_threshold = df_thresholds.loc[best_idx, "Threshold"]

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best F1: {df_thresholds.loc[best_idx, 'F1']:.4f}")

# Mostrar los mejores thresholds
display(
    df_thresholds.sort_values(
        "F1", ascending=False
    ).head(10)
)


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    df_thresholds["Threshold"],
    df_thresholds["Precision"],
    label="Precision"
)

plt.plot(
    df_thresholds["Threshold"],
    df_thresholds["Recall"],
    label="Recall"
)

plt.plot(
    df_thresholds["Threshold"],
    df_thresholds["F1"],
    label="F1-score",
    linewidth=2
)

plt.axvline(
    best_threshold,
    color="red",
    linestyle="--",
    label=f"Best threshold = {best_threshold:.2f}"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Threshold Optimization - XGBoost")

plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Final Holdout Evaluation

After choosing the model and threshold using training data, fit XGBoost on all of `X_train` and evaluate on `X_test`. Do not retune the threshold based on these test results.

In [ ]:
# Fit the final model
xgb_cv_pipeline.fit(X_train, y_train)

# Get test probabilities
y_proba_test = xgb_cv_pipeline.predict_proba(X_test)[:, 1]

# Apply the optimized threshold
y_pred_test = (
    y_proba_test >= best_threshold
).astype(int)

# Final evaluation
print("XGBoost - Final Evaluation")

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_proba_test), 4)
)

print(
    "\nClassification Report:"
)

print(
    classification_report(y_test, y_pred_test)
)

print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(y_test, y_pred_test)
)

## Conclusions and Limitations

- Compare cross-validation ROC-AUC means and standard deviations before selecting the final model.
- Report the selected out-of-fold threshold alongside the holdout classification metrics.
- **Important limitation:** during earlier development, test results from several experiments were inspected. As a result, the reported test metrics are not a fully untouched, unbiased estimate of future performance.
- Future work: evaluate on fresh data, add business-oriented cost analysis, and validate the workflow before deployment.